# Exploratory Data Analysis — Framingham Heart Study

This notebook explores feature distributions, missing values, class balance, and predictor relationships before any preprocessing.

**Dataset:** Framingham Heart Study | **Target:** `TenYearCHD` (binary) | **Task:** Binary Classification

## 1. Imports

In [ ]:
!pip install missingno -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
%matplotlib inline

# Global plot style
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 20)

## 2. Data Loading & Inspection

**Source:** Framingham Heart Study — Kaggle / NHLBI  
**Format:** CSV  
**File path:** `../data/raw/framingham.csv`

In [ ]:
df = pd.read_csv('../data/raw/framingham.csv')
print('Shape:', df.shape)
df.head()

## 3. Data Types & Structure

Inspecting column types, non-null counts, and summary statistics to understand the structure of the dataset.

In [ ]:
df.info()

In [ ]:
df.describe().T.round(2)

**Observations:**
- 4,240 rows and 16 columns (15 features + 1 target)
- Mix of `int64` (binary/count features) and `float64` (continuous clinical measurements)
- Several columns have missing values — addressed in Section 4
- `TenYearCHD` is the binary target: `0` = no CHD risk, `1` = CHD risk within 10 years

## 4. Missing Value Analysis

Identifying which features have missing data and the extent of missingness before any imputation.

In [ ]:
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Count': missing, 'Pct %': pct})
miss_df = miss_df[miss_df['Count'] > 0].sort_values('Pct %', ascending=False)
print(miss_df)

In [ ]:
# Horizontal bar chart — easy to read missing % per feature
plt.figure(figsize=(7, 3.5))
miss_df['Pct %'].plot(kind='barh', color='steelblue', edgecolor='white')
plt.xlabel('% Missing')
plt.title('Features with Missing Values')
plt.tight_layout()
plt.show()

In [ ]:
# Missingno matrix — shows the pattern / co-occurrence of missingness
msno.matrix(df)
plt.title('Missing Values Matrix')
plt.show()

**Key findings:**
- `glucose` has the most missing data at **9.15%** — the most impactful column to address
- All other columns with missing values are **< 3%**
- `TenYearCHD` has **zero** missing values — target is complete
- **Plan:** Median imputation for all missing columns (applied in `02_preprocessing.ipynb`)

## 5. Target Variable Distribution

Visualizing the class balance of `TenYearCHD` to understand the degree of class imbalance.

In [ ]:
counts = df['TenYearCHD'].value_counts().sort_index()
print(counts)
print(f'\nClass imbalance: {counts[1]/len(df)*100:.1f}% positive cases')

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['No CHD (0)', 'CHD Risk (1)'], counts.values,
       color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.5)
ax.set_ylabel('Count')
ax.set_title('Target Class Distribution')
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

**Key finding:**
- **~85% No CHD / ~15% CHD Risk** — significant class imbalance
- A naive model that always predicts 'No CHD' would achieve ~85% accuracy while being clinically useless
- **Plan:** Use SMOTE on the training set to balance classes; use **AUC-ROC and F1-score** as primary metrics (not accuracy)

## 6. Univariate Feature Distributions

Histograms for all numeric features to understand the distribution shape, range, and potential skew.

In [ ]:
num_cols = [c for c in df.select_dtypes(include='number').columns if c != 'TenYearCHD']
ncols, nrows = 4, -(-len(num_cols) // 4)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=25, color='#4C72B0', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

**Observations:**
- `sysBP`, `diaBP`, `totChol`, `BMI` are approximately normally distributed
- `cigsPerDay` and `glucose` are right-skewed with many zero/low values
- Binary features (`male`, `currentSmoker`, `diabetes`, `prevalentHyp`, `prevalentStroke`, `BPMeds`) show as two-bar distributions
- No extreme outliers that would require removal, but scaling is needed before modeling

## 7. Bivariate Analysis — Features vs Target

Boxplots comparing each feature's distribution between CHD-negative (0) and CHD-positive (1) patients.

In [ ]:
nrows = -(-len(num_cols) // 4)
fig, axes = plt.subplots(nrows, 4, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x='TenYearCHD', y=col, data=df, ax=axes[i],
                hue='TenYearCHD', palette={0: '#4C72B0', 1: '#DD8452'},
                legend=False)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('CHD Risk')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by CHD Outcome', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

**Key findings:**
- `age`, `sysBP`, `diaBP`, `glucose` show the clearest separation between CHD groups — strong predictors
- `prevalentHyp` patients have notably higher CHD rates — clinically expected
- `heartRate`, `currentSmoker`, `education` show minimal separation — weak predictors
- Tree-based models (Random Forest, XGBoost) will naturally de-emphasize weak features

## 8. Correlation Heatmap

Pairwise Pearson correlations across all features to identify multicollinearity and target predictors.

In [ ]:
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))   # show lower triangle only

plt.figure(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.4, vmin=-1, vmax=1)
plt.title('Pairwise Correlations (lower triangle)')
plt.tight_layout()
plt.show()

In [ ]:
# Sorted absolute correlations with the target
target_corr = corr['TenYearCHD'].drop('TenYearCHD').abs().sort_values(ascending=False)
print('Absolute correlation with TenYearCHD:')
print(target_corr.round(3))

**Key findings:**
- **Top predictors:** `age` (0.225), `sysBP` (0.216), `prevalentHyp` (0.177), `diaBP` (0.145), `glucose` (0.126)
- **Weak predictors:** `heartRate` (0.023), `currentSmoker` (0.019) — near-zero correlation
- **Multicollinearity note:** `sysBP` and `diaBP` are correlated (r ≈ 0.79) — expected clinically
- All features will be kept: tree-based models handle irrelevant features naturally; LR/SVM rely on scaling

## 9. EDA Summary

| Finding | Detail |
|---------|--------|
| Rows / Columns | 4,240 × 16 |
| Missing values | `glucose` ~9.2%, `education` ~2.5%, others < 2% |
| Class imbalance | ~85% negative / ~15% positive |
| Strongest predictors | `age`, `sysBP`, `prevalentHyp`, `diaBP`, `glucose` |
| Weak predictors | `heartRate`, `currentSmoker`, `education` |
| Evaluation plan | AUC-ROC and F1-score as primary metrics |

### Preliminary Preprocessing Plan
1. **Median imputation** — fill all missing values with column medians
2. **Stratified 70/15/15 split** — preserve 85/15 class ratio across all sets
3. **StandardScaler** — fit on training set only, transform val & test
4. **SMOTE** — oversample minority class in training set only

All preprocessing steps are implemented in `02_preprocessing.ipynb`.